# Cosmos Chimera Forge — Drive-only Colab (PR #26)

本 Notebook 僅從 Google Drive 載入已發布的 `anime3d_colab_source.zip`，不呼叫 GitHub。開始時只做 CPU preflight，不啟動 GPU、付費 API，亦不自動建立人工核准紀錄。

GPU 模型、SkinTokens 與 Blender 必須在相容環境獨立安裝。初次使用先設定真實 `DRIVE_ROOT`、`SUBJECT_ID` 與 Drive 角色 spec；檢查報告內的阻擋原因，不可虛構人工核准。


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import json, os, subprocess, sys
drive.mount('/content/drive')
auth.authenticate_user()
# 編輯下列路徑，必須是自己的 Drive 掛載路徑，不接受 GitHub fallback。
DRIVE_ROOT = Path('/content/drive/MyDrive/project/cosmos_chimera_forge')
SUBJECT_ID = '<subject_id>'
SPEC = DRIVE_ROOT / 'characters' / SUBJECT_ID / '00_settings' / 'asset_spec.json'
assert DRIVE_ROOT.is_dir(), f'Drive folder not mounted: {DRIVE_ROOT}'
assert SPEC.is_file(), f'Missing real character asset_spec: {SPEC}'
assert SUBJECT_ID != '<subject_id>', 'Set SUBJECT_ID to a real Drive character ID'


In [ ]:
# Read only the pinned offline source archive on Google Drive. No clone and no network fetch.
import hashlib, zipfile, py_compile
PACKAGE = DRIVE_ROOT / 'pipeline' / 'character_colab_runtime_PR26'
ARCHIVE = PACKAGE / 'anime3d_colab_source.zip'
INTEGRITY = PACKAGE / 'bundle_integrity.json'
assert ARCHIVE.is_file() and INTEGRITY.is_file(), 'Offline runtime files missing on Drive'
pinned = json.loads(INTEGRITY.read_text(encoding='utf8'))
assert hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() == pinned['archive_sha256'], 'Runtime archive hash mismatch'
REPO = Path('/content/cosmos_chimera_forge')
REPO.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as bundle:
    assert set(bundle.namelist()) == set(pinned['file_sha256']), 'Archive contents differ from pinned manifest'
    for name in bundle.namelist():
        dest=(REPO / name).resolve()
        assert dest.is_relative_to(REPO.resolve()) and not name.startswith('/'), 'Unsafe runtime path'
        data=bundle.read(name)
        assert hashlib.sha256(data).hexdigest()==pinned['file_sha256'][name], 'Tampered runtime source: '+name
        dest.parent.mkdir(parents=True,exist_ok=True)
        dest.write_bytes(data)
        if name.endswith('.py'): py_compile.compile(str(dest),doraise=True)
print('Verified Drive-only runtime:',len(pinned['file_sha256']),'files; PR #26 snapshot',pinned['source_head_sha'])


In [ ]:
# CPU plan + preflight; no fabricated approval, no GPU generation by default.
import shutil
DRIVE_PROJECT = DRIVE_ROOT / 'characters'
spec = json.loads(SPEC.read_text(encoding='utf8'))
ASSET_ID = spec['asset_id']
assert spec.get('subject_id') == SUBJECT_ID
assert spec.get('character_colab_workflow') == 'character-colab/2'
STAGE = 'pre_gpu'  # pre_gpu -> gpu -> independent appearance_review -> post_gpu
EXECUTE = False
BACKEND = 'pixal3d'
RIG_BACKEND = 'none'  # enable skintokens only after isolated runtime install
BACKEND_CONFIG = ''  # e.g. Drive path to installed backend config JSON
BLENDER = shutil.which('blender') or ''
JOB = DRIVE_PROJECT / SUBJECT_ID / '_03a_colab' / ASSET_ID
cmd=[sys.executable,str(REPO/'scripts/anime3d_preflight.py'),'--root',str(DRIVE_PROJECT),'--spec',str(SPEC),'--asset-id',ASSET_ID,'--backend',BACKEND,'--stage',STAGE,'--report',str(JOB/'qa/preflight.json')]
if BACKEND_CONFIG: cmd += ['--backend-config',BACKEND_CONFIG]
if RIG_BACKEND!='none':cmd += ['--rig-backend',RIG_BACKEND]
if BLENDER:cmd += ['--blender',BLENDER]
check=subprocess.run(cmd,check=False)
if check.returncode: raise RuntimeError('Preflight BLOCKED. Inspect the report; no GPU call was made.')
print('CPU preflight passed. EXECUTE remains',EXECUTE)


In [ ]:
# Run stage only after explicit authorization and preflight.
assert STAGE in ('pre_gpu','gpu','post_gpu')
run=[sys.executable,str(REPO/'scripts/anime3d_universal.py'),'--root',str(DRIVE_PROJECT),'--spec',str(SPEC),'--asset-id',ASSET_ID,'--backend',BACKEND,'--pipeline','colab','--stage',STAGE]
if BACKEND_CONFIG:run += ['--backend-config',BACKEND_CONFIG]
if RIG_BACKEND!='none':run += ['--rig-backend',RIG_BACKEND]
if BLENDER and STAGE=='post_gpu':run += ['--blender',BLENDER]
if EXECUTE:run += ['--execute']
# Don't infer REVIEW_REQUIRED from a Colab exception: inspect typed_state.json after the run.
# For actual gpu execution, only use validated guard shutdown procedure in docs.
if EXECUTE and STAGE=='gpu':
    from google.colab import runtime
    try:subprocess.run(run,check=True)
    finally:
        from sys import path as sys_path
        sys_path.insert(0,str(REPO/'scripts'))
        from anime3d_colab_guard import shutdown_colab_runtime
        shutdown_colab_runtime('gpu_stage_finished',root=DRIVE_PROJECT)
else:subprocess.run(run,check=True)


## 外觀品質審查與後續執行

`STAGE=gpu` 只建立來源 GLB；在外觀審查報告 `qa/appearance_review.json` 綁定模型／參考圖雜湊、五項品質分數並達到門檻之前，不得進入綁骨。`post_gpu` 如果使用 SkinTokens 仍可能需要 GPU。MV-Adapter、Texture Refinement 與自動外觀評分在 PR #26 尚未整合，不可宣稱已執行。

來源、模型、backend 改變必須新建 `asset_id`。請參照 ZIP 中的 `docs/pipeline/CHARACTER_COLAB_QUALITY.md`。
